# TRIAGE-EG Stage 1D — Vietnamese Translation Bridge Ablation

Stage 1C is a frozen baseline. This notebook generates only `VI_TRANSLATED_EN`; it does not regenerate `EN_DIRECT` or `VI_DIRECT`, rerank results, or rebuild the Stage 1A index. Human review is required before any language-bridge quality conclusion.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path(os.environ.get("AIC_REPO_DIR", "/kaggle/working/AIC2026_TeamPTK_SGU"))
REFRESH_REPO = os.environ.get("AIC_REFRESH_REPO", "0") == "1"
DATA_ROOT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
STAGE0_ROOT = Path(os.environ.get("AIC_STAGE0_ROOT", "/kaggle/working/triage_eg_stage0_audit"))
STAGE0_BUNDLE = os.environ.get(
    "AIC_STAGE0_BUNDLE", "/kaggle/input/datasets/irthn1311/triage-eg-stage0-audit-bundle"
)
STAGE1_ROOT = Path(
    os.environ.get(
        "AIC_STAGE1_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-input-bundle"
    )
)
STAGE1B_ROOT = Path(
    os.environ.get(
        "AIC_STAGE1B_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports",
    )
)
STAGE1C_ROOT = Path(
    os.environ.get(
        "AIC_STAGE1C_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-stage1c-qualitative-eval-bundle",
    )
)
CLIP_ASSET_ROOT = Path(
    os.environ.get(
        "AIC_OPENAI_CLIP_ASSET_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32"
    )
)
OPUS_ASSET_ROOT = Path(
    os.environ.get(
        "AIC_OPUS_MT_ASSET_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-opus-mt-vi-en"
    )
)
OUTPUT_ROOT = Path(
    os.environ.get(
        "AIC_STAGE1D_OUTPUT_ROOT", "/kaggle/working/triage_eg_stage1d_translation_ablation"
    )
)
CONFIG_VALUE = os.environ.get(
    "AIC_STAGE1D_CONFIG", "configs/retrieval/stage1d_translation_ablation.yaml"
)
TRANSLATOR_DEVICE = os.environ.get("AIC_STAGE1D_TRANSLATOR_DEVICE", "cpu")
CLIP_DEVICE = os.environ.get("AIC_STAGE1D_CLIP_DEVICE", "auto")
print(
    {
        "ref": REPO_REF,
        "stage1c": str(STAGE1C_ROOT),
        "opus": str(OPUS_ASSET_ROOT),
        "output": str(OUTPUT_ROOT),
    }
)

In [ ]:
def git_result(*args, cwd=None):
    return subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, check=False)


def git(*args, cwd=None):
    result = git_result(*args, cwd=cwd)
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout.strip()


if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout")
if not (REPO_DIR / ".git").is_dir():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    git("clone", "--filter=blob:none", "--no-checkout", REPO_URL, str(REPO_DIR))
target = None
if not REFRESH_REPO:
    for candidate in (REPO_REF, f"origin/{REPO_REF}"):
        probe = git_result("rev-parse", "--verify", f"{candidate}^{{commit}}", cwd=REPO_DIR)
        if probe.returncode == 0:
            target = probe.stdout.strip()
            break
if target is None:
    git("fetch", "--depth", "1", "origin", REPO_REF, cwd=REPO_DIR)
    target = "FETCH_HEAD"
git("checkout", "--detach", target, cwd=REPO_DIR)
COMMIT = git("rev-parse", "HEAD", cwd=REPO_DIR)
os.environ["AIC_RESOLVED_GIT_COMMIT"] = COMMIT
sys.path.insert(0, str(REPO_DIR / "src"))
print("resolved commit:", COMMIT)

In [ ]:
from dataclasses import replace

from triage_eg.retrieval.stage1.stage0_loader import resolve_stage0_root
from triage_eg.retrieval.stage1b.inputs import resolve_stage1_root
from triage_eg.retrieval.stage1c.inputs import resolve_stage1b_root
from triage_eg.retrieval.stage1d import Stage1DConfig, resolve_input_root, settings_from_yaml
from triage_eg.retrieval.stage1d.inputs import STAGE1C_REQUIRED, TRANSLATOR_REQUIRED

STAGE0_ROOT = resolve_stage0_root(
    STAGE0_ROOT,
    bundle_path=STAGE0_BUNDLE,
    search_root=Path("/kaggle/input"),
    excluded_roots=(DATA_ROOT,),
)
STAGE1_ROOT = resolve_stage1_root(
    STAGE1_ROOT,
    search_root=Path("/kaggle/input"),
    excluded_roots=(DATA_ROOT, STAGE0_ROOT),
    materialize_root=Path("/kaggle/working/triage_eg_stage1d_stage1_input"),
)
STAGE1B_ROOT = resolve_stage1b_root(
    STAGE1B_ROOT,
    search_root=Path("/kaggle/input"),
    materialize_root=Path("/kaggle/working/triage_eg_stage1d_stage1b_input"),
)
STAGE1C_ROOT, STAGE1C_MODE = resolve_input_root(
    STAGE1C_ROOT,
    required=STAGE1C_REQUIRED,
    search_root=Path("/kaggle/input"),
    materialize_root=Path("/kaggle/working/triage_eg_stage1c_frozen_runtime"),
    archive_keyword="stage1c",
)
OPUS_ASSET_ROOT, OPUS_MODE = resolve_input_root(
    OPUS_ASSET_ROOT,
    required=TRANSLATOR_REQUIRED,
    search_root=Path("/kaggle/input"),
    materialize_root=Path("/kaggle/working/triage_eg_opus_mt_vi_en_runtime"),
    archive_keyword="opus-mt-vi-en",
)
if not CLIP_ASSET_ROOT.is_dir():
    fallback = Path("/kaggle/input/aic2026-openai-clip-vit-b32")
    if fallback.is_dir():
        CLIP_ASSET_ROOT = fallback
CONFIG_PATH = Path(CONFIG_VALUE)
CONFIG_PATH = CONFIG_PATH if CONFIG_PATH.is_absolute() else REPO_DIR / CONFIG_PATH
translator_cfg, generation_cfg, retrieval_cfg, review_cfg = settings_from_yaml(CONFIG_PATH)

translator_cfg = replace(translator_cfg, device=TRANSLATOR_DEVICE)
CONFIG = Stage1DConfig(
    repo_root=REPO_DIR,
    dataset_root=DATA_ROOT,
    stage0_root=STAGE0_ROOT,
    stage1_root=STAGE1_ROOT,
    stage1b_root=STAGE1B_ROOT,
    stage1c_root=STAGE1C_ROOT,
    clip_asset_root=CLIP_ASSET_ROOT,
    translator_asset_root=OPUS_ASSET_ROOT,
    output_root=OUTPUT_ROOT,
    translator=translator_cfg,
    generation=generation_cfg,
    retrieval=retrieval_cfg,
    review=review_cfg,
    clip_device=CLIP_DEVICE,
    overwrite=True,
    strict_root=True,
    build_git_commit=COMMIT,
    stage1c_materialization=STAGE1C_MODE,
    translator_asset_materialization=OPUS_MODE,
)
print(
    {
        "stage0": str(STAGE0_ROOT),
        "stage1": str(STAGE1_ROOT),
        "stage1b": str(STAGE1B_ROOT),
        "stage1c": str(STAGE1C_ROOT),
        "clip": str(CLIP_ASSET_ROOT),
        "opus": str(OPUS_ASSET_ROOT),
    }
)

In [ ]:
from triage_eg.retrieval.stage1d import preflight_stage1d

PREFLIGHT = preflight_stage1d(CONFIG)
print(
    json.dumps(
        {
            k: PREFLIGHT[k]
            for k in (
                "stage1c_frozen_baseline_status",
                "stage1c_query_suite_fingerprint",
                "stage1_index_fingerprint",
                "pairs_selected",
                "baseline_regenerated",
            )
        },
        indent=2,
    )
)
assert PREFLIGHT["baseline_regenerated"] is False

In [ ]:
print(
    json.dumps(
        {
            k: PREFLIGHT[k]
            for k in (
                "translator_asset_status",
                "translator_revision",
                "translator_hash_status",
                "translator_dependencies",
                "translator_device",
                "offline_only",
            )
        },
        indent=2,
    )
)

In [ ]:
from triage_eg.retrieval.stage1d.inputs import validate_translator_asset
from triage_eg.retrieval.stage1d.translator import OfflineViEnTranslator

translator_asset = validate_translator_asset(OPUS_ASSET_ROOT)
TRANSLATOR_INSTANCE = OfflineViEnTranslator(
    translator_asset["model_root"], CONFIG.translator, CONFIG.generation
).load()
print(json.dumps(TRANSLATOR_INSTANCE.runtime_manifest(), indent=2, default=str))

In [ ]:
from triage_eg.retrieval.stage1b.assets import load_multimodal_encoder
from triage_eg.retrieval.stage1c.runner import _runtime_candidate
from triage_eg.retrieval.stage1d.runner import _validate_stage1d_inputs

validated_inputs = _validate_stage1d_inputs(CONFIG)
runtime_candidate = _runtime_candidate(
    validated_inputs["stage1c_config"],
    validated_inputs["selected_contract"],
    OUTPUT_ROOT.parent / ".stage1d_clip_runtime",
)
CLIP_ENCODER = load_multimodal_encoder(runtime_candidate)
print(
    "verified encoder:",
    runtime_candidate.candidate_id,
    "checkpoint:",
    runtime_candidate.checkpoint_sha256,
)

In [ ]:
import pandas as pd

BASELINE = validated_inputs["baseline"]
pair_table = [
    {"pair_id": p, "en_reference": q["en"].text, "vi_original": q["vi"].text}
    for p, q in sorted(BASELINE.pairs.items())
]
display(pd.DataFrame(pair_table))

In [ ]:
from triage_eg.retrieval.stage1d import run_stage1d

RESULT = run_stage1d(
    CONFIG,
    translator_factory=lambda *_: TRANSLATOR_INSTANCE,
    clip_adapter_factory=lambda _: CLIP_ENCODER,
)
translations = [
    json.loads(line)
    for line in (OUTPUT_ROOT / "translations/translations.jsonl")
    .read_text(encoding="utf-8")
    .splitlines()
]
display(
    pd.DataFrame(translations)[
        ["pair_id", "original_vi_text", "translated_text_for_clip", "status"]
    ]
)

In [ ]:
comparisons = [
    json.loads(line)
    for line in (OUTPUT_ROOT / "comparisons/pair_comparisons.jsonl")
    .read_text(encoding="utf-8")
    .splitlines()
]
clip_rows = [{"pair_id": x["pair_id"], **x["text_space"]} for x in comparisons]
display(pd.DataFrame(clip_rows))

In [ ]:
summary = RESULT.summary
print(json.dumps(summary["retrieval"], indent=2))
assert summary["retrieval"]["baseline_retrieval_source"] == "FROZEN_STAGE1C_ARTIFACTS"
assert summary["retrieval"]["ranking_policy"] == "RAW_STAGE1A_EXACT_COSINE_NO_RERANKING"

In [ ]:
movement_rows = [{"pair_id": x["pair_id"], **x["ranking_alignment"]} for x in comparisons]
display(pd.json_normalize(movement_rows))

In [ ]:
print(json.dumps(summary["structural_diagnostics"], indent=2, ensure_ascii=False))
print("Structural warnings are diagnostics; ranking was not changed.")

In [ ]:
from IPython.display import Image as DisplayImage
from IPython.display import display

for pair_id in [x["pair_id"] for x in comparisons[:5]]:
    sheet = OUTPUT_ROOT / "comparisons" / pair_id / "comparison_top5.jpg"
    if sheet.is_file():
        print(pair_id)
        display(DisplayImage(filename=str(sheet), width=1200))

In [ ]:
from triage_eg.retrieval.stage1d import patch_blinded_review_visuals

PATCH = patch_blinded_review_visuals(OUTPUT_ROOT, DATA_ROOT)
summary = json.loads((OUTPUT_ROOT / "stage1d_summary.json").read_text(encoding="utf-8"))
review_csv = OUTPUT_ROOT / "review/review_template_blinded.csv"
print("review CSV:", review_csv)
print("expected judgments:", summary["human_review"]["judgments_expected"])
print("blinded:", summary["human_review"]["blinded"])
print("blinded sheets:", PATCH["blinded_sheet_count"])

In [ ]:
print("STAGE1D_EXECUTION =", summary["execution_status"])
print("TRANSLATOR =", summary["translator"]["asset_status"])
print(
    "TRANSLATED_RETRIEVAL =",
    "COMPLETE" if summary["retrieval"]["translated_queries_failed"] == 0 else "FAILED",
)
print("HUMAN_REVIEW = NOT_REVIEWED")
print("BLINDED_REVIEW_SHEETS =", summary["human_review"]["blinded_sheet_count"])
print("REVIEW_ROWS =", summary["human_review"]["judgments_expected"])
print("FORMAL_HUMAN_REVIEW_EXECUTABILITY = READY")
print("LANGUAGE_BRIDGE_QUALITY_STATUS = NOT_REVIEWED")

In [ ]:
from zipfile import ZipFile

from triage_eg.retrieval.stage1d import create_stage1d_bundle

zip_path = Path("/kaggle/working/triage_eg_stage1d_translation_ablation_bundle.zip")
create_stage1d_bundle(OUTPUT_ROOT, zip_path)
with ZipFile(zip_path) as archive:
    members = archive.namelist()
assert any(name.endswith("ranked_frames.jsonl") for name in members)
assert any(name.endswith("comparison_top5.jpg") for name in members)
assert "review/blinded_sheet_index.csv" in members
assert (
    len(
        [
            name
            for name in members
            if name.startswith("review/blinded_sheets/") and name.endswith(".jpg")
        ]
    )
    == 14
)
assert not any(
    name.endswith((".pt", ".pth", ".bin", ".npy", ".mp4", ".avi", ".mkv"))
    or name.startswith("logs/")
    for name in members
)
print("DOWNLOAD ZIP:", zip_path, "size_bytes=", zip_path.stat().st_size, "members=", len(members))